In [2]:
from nltk.corpus import gutenberg

docs = gutenberg.fileids()
raw_docs = [gutenberg.raw(file_id) for file_id in docs]

def split_into_chunks(text, chunk_size=2000):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

all_docs = []
for text in raw_docs:
    chunks = split_into_chunks(text)
    all_docs.extend(chunks)

documents = all_docs[:20]   # take first 20 documents
len(documents)


20

In [3]:
import nltk
import re
from nltk.corpus import stopwords

# Your custom stopwords (at least 5)
my_stopwords = {"chapter", "project", "gutenberg", "said", "would"}

# Combine with NLTK stopwords
stop_words = set(stopwords.words("english")).union(my_stopwords)

def preprocess(doc):
    # Lowercase
    doc = doc.lower()
    
    # Remove punctuation
    doc = re.sub(r"[^a-z\s]", "", doc)
    
    # Tokenize
    tokens = nltk.word_tokenize(doc)
    
    # Remove stopwords and short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    return tokens

processed_docs = [preprocess(doc) for doc in documents]


In [15]:
for i, doc in enumerate(processed_docs):
    print(f"Document {i+1} terms:", doc[:30], "...\n")


Document 1 terms: ['emma', 'jane', 'austen', 'volume', 'emma', 'woodhouse', 'handsome', 'clever', 'rich', 'comfortable', 'home', 'happy', 'disposition', 'seemed', 'unite', 'best', 'blessings', 'existence', 'lived', 'nearly', 'twentyone', 'years', 'world', 'little', 'distress', 'vex', 'youngest', 'two', 'daughters', 'affectionate'] ...

Document 2 terms: ['sometimes', 'fanciful', 'troublesome', 'dearest', 'papa', 'think', 'could', 'mean', 'suppose', 'knightley', 'mean', 'horrible', 'idea', 'meant', 'knightley', 'loves', 'find', 'fault', 'know', 'jokeit', 'joke', 'always', 'say', 'like', 'one', 'another', 'knightley', 'fact', 'one', 'people'] ...

Document 3 terms: ['every', 'probability', 'greater', 'happiness', 'yet', 'passed', 'never', 'unhappy', 'man', 'temper', 'secured', 'even', 'first', 'marriage', 'second', 'must', 'shew', 'delightful', 'welljudging', 'truly', 'amiable', 'woman', 'could', 'must', 'give', 'pleasantest', 'proof', 'great', 'deal', 'better'] ...

Document 4 terms: ['

In [5]:
from collections import Counter
import math

# Build vocabulary
vocab = sorted(set(term for doc in processed_docs for term in doc))
vocab[:20]
 
tf = []

for doc in processed_docs:
    counts = Counter(doc)
    total = len(doc)
    tf.append({term: counts[term] / total for term in vocab})

df = {term: 0 for term in vocab}

for term in vocab:
    for doc in processed_docs:
        if term in doc:
            df[term] += 1


N = len(processed_docs)

idf = {term: math.log(N / df[term]) for term in vocab}

#Compute TF-IDF Matrix
tfidf = []

for d in range(N):
    tfidf.append({term: tf[d][term] * idf[term] for term in vocab})

query = ["love", "truth", "life"]
query = [t for t in query if t in vocab]
scores = []

for d in range(N):
    score = sum(tfidf[d][t] for t in query)
    scores.append((d, score))
    
sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
sorted_scores


[(13, 0.00390359619592448),
 (15, 0.0025875836199264392),
 (1, 0.002580108273110142),
 (5, 0.0023731561672574956),
 (11, 0.0019221519317490924),
 (19, 0.0017757796152115678),
 (16, 0.0016797915279028344),
 (3, 0.0016178469379367864),
 (9, 0.00138475125127211),
 (6, 0.0013553925674995562),
 (7, 0.0013334047390580804),
 (17, 0.0013181309161593624),
 (2, 0.0012074798423999196),
 (4, 0.0010385634384540824),
 (12, 0.0010360698887819238),
 (10, 0.0010189447666532968),
 (14, 0.0009852125768896605),
 (0, 0.0006472037625461887),
 (18, 0.0006407173105830308),
 (8, 0.000345771721696852)]

In [6]:
import numpy as np

# Convert tf-idf dictionaries to vectors
tfidf_vectors = []

for d in range(N):
    vec = np.array([tfidf[d][term] for term in vocab])
    tfidf_vectors.append(vec)

def cosine_similarity(v1, v2):
    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0
    return dot / (norm1 * norm2)

similarities = []

for i in range(N):
    for j in range(i+1, N):
        sim = cosine_similarity(tfidf_vectors[i], tfidf_vectors[j])
        similarities.append((i, j, sim))

# Sort by similarity descending
similarities_sorted = sorted(similarities, key=lambda x: x[2], reverse=True)
similarities_sorted[:10]   # top 10 most similar pairs


[(11, 12, np.float64(0.1757297378696896)),
 (6, 7, np.float64(0.17417028378390526)),
 (9, 10, np.float64(0.17045760070062532)),
 (0, 1, np.float64(0.16884371218204106)),
 (17, 18, np.float64(0.1631175995326115)),
 (1, 19, np.float64(0.15793008751911414)),
 (3, 9, np.float64(0.1431221946466698)),
 (1, 15, np.float64(0.14281102227299258)),
 (4, 9, np.float64(0.13739010742564245)),
 (0, 15, np.float64(0.1358814298090162))]

In [9]:
from nltk.util import ngrams

def preprocess_with_bigrams(doc):
    # Lowercase
    doc = doc.lower()
    
    # Remove punctuation
    doc = re.sub(r"[^a-z\s]", "", doc)
    
    # Tokenize
    tokens = nltk.word_tokenize(doc)
    
    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    # Create bigrams
    bigrams = ["_".join(bg) for bg in ngrams(tokens, 2)]
    return tokens + bigrams
    
    # Return unigrams + bigrams
    return tokens + bigrams

processed_docs_mwe = [preprocess_with_bigrams(doc) for doc in documents]
processed_docs_mwe[0][:30]


['emma',
 'jane',
 'austen',
 'volume',
 'emma',
 'woodhouse',
 'handsome',
 'clever',
 'rich',
 'comfortable',
 'home',
 'happy',
 'disposition',
 'seemed',
 'unite',
 'best',
 'blessings',
 'existence',
 'lived',
 'nearly',
 'twentyone',
 'years',
 'world',
 'little',
 'distress',
 'vex',
 'youngest',
 'two',
 'daughters',
 'affectionate']

In [10]:
# Build vocabulary with MWEs
vocab_mwe = sorted(set(term for doc in processed_docs_mwe for term in doc))

# Compute TF
tf_mwe = []
for doc in processed_docs_mwe:
    counts = Counter(doc)
    total = len(doc)
    tf_mwe.append({term: counts[term] / total for term in vocab_mwe})

# Compute DF
df_mwe = {term: 0 for term in vocab_mwe}
for term in vocab_mwe:
    for doc in processed_docs_mwe:
        if term in doc:
            df_mwe[term] += 1

# Compute IDF
idf_mwe = {term: math.log(N / df_mwe[term]) for term in vocab_mwe}

# Compute TF-IDF
tfidf_mwe = []
for d in range(N):
    tfidf_mwe.append({term: tf_mwe[d][term] * idf_mwe[term] for term in vocab_mwe})

list(tfidf_mwe[0].items())[:10]



[('abbey', 0.0),
 ('abbey_holidays', 0.0),
 ('abbey_knightley', 0.0),
 ('abbey_never', 0.0),
 ('abbey_parish', 0.0),
 ('abbey_thank', 0.0),
 ('abbey_tomorrow', 0.0),
 ('abbey_two', 0.0),
 ('abbeymill', 0.0),
 ('abbeymill_farm', 0.0)]

In [ ]:
tweets = [
    "Loving the new features in #Python! Great work @devteam",
    "Big news in AI today — huge breakthrough in #MachineLearning",
    "Had a great chat with @OpenAI team about future research",
    "Working on a new project involving deep learning and #AI",
    "Can't believe the results from our latest experiment #AI #Research"
]
import re
import nltk
from nltk.corpus import stopwords

tweet_stopwords = set(stopwords.words("english"))

def preprocess_tweet(text):
    text = text.lower()
    # keep letters, digits, @ and #, and spaces
    tokens = re.findall(r"#\w+|@\w+|\w+", text)
   
    tokens = [t for t in tokens if t not in tweet_stopwords]
    return tokens

processed_tweets = [preprocess_tweet(t) for t in tweets]
processed_tweets

def social_boost(term):
    if term.startswith("#"):
        return 2.0      
    if term.startswith("@"):
        return 1.5      
    if term.isupper():
        return 1.3     
    return 1.0          

tweet_vocab = sorted(set(term for doc in processed_tweets for term in doc))
tweet_vocab

tweet_tf = []
for doc in processed_tweets:
    tweet_tf.append({term: (1 if term in doc else 0) for term in tweet_vocab})

import math

T = len(processed_tweets)

tweet_df = {term: sum(1 for doc in processed_tweets if term in doc) for term in tweet_vocab}
tweet_idf = {term: math.log(T / tweet_df[term]) for term in tweet_vocab}
29
social_tfidf = []

for d in range(T):
    weights = {}
    for term in tweet_vocab:
        weights[term] = tweet_tf[d][term] * tweet_idf[term] * social_boost(term)
    social_tfidf.append(weights)

# Top 10 weighted terms for tweet 0
sorted(social_tfidf[0].items(), key=lambda x: x[1], reverse=True)[:10]


[('#python', 3.2188758248682006),
 ('@devteam', 2.4141568686511503),
 ('features', 1.6094379124341003),
 ('loving', 1.6094379124341003),
 ('work', 1.6094379124341003),
 ('great', 0.9162907318741551),
 ('new', 0.9162907318741551),
 ('#ai', 0.0),
 ('#machinelearning', 0.0),
 ('#research', 0.0)]